In [1]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 994.0/994.0 kB 41.9 MB/s eta 0:00:00
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.3.88
    Uninstalling ultralytics-8.3.88:
      Successfully uninstalled ultralytics-8.3.88


In [ ]:
import os
import yaml
import cv2
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
from ultralytics import YOLO

# 📂 Configuração do dataset BDD100K
dataset_yaml = {
    "train": "/Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/Test/yolo_dataset/images/train",
    "val": "/Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/Test/yolo_dataset/images/validation",
    "test": "/Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/Test/yolo_dataset/images/test",
    "nc": 10,
    "names": [
        "traffic sign", "traffic light", "car", "rider", "motor",
        "person", "bus", "truck", "bike", "train"
    ]
}

# 📝 Salva em dataset.yaml
with open("bdd100k_dataset.yaml", "w") as f:
    yaml.dump(dataset_yaml, f)


# 🚀 Treinamento YOLOv8
model = YOLO("yolov8n.pt")  # ou yolov8m.pt etc.

model.train(
    data="bdd100k_dataset.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    name="bdd100k_finetuned",
    workers=2,
    patience=10,
    device="cpu"  # Altere para "0" se estiver com GPU
)


# 📍 Carregar melhor modelo treinado
best_model_path = "runs/detect/bdd100k_finetuned/weights/best.pt"
model = YOLO(best_model_path)

# 📊 Avaliação no conjunto de validação
results = model.val(data="bdd100k_dataset.yaml", split="val", save_json=True)


# 🧪 Visualização de predições
val_dir = dataset_yaml["val"]
image_paths = sorted([
    os.path.join(val_dir, f) for f in os.listdir(val_dir)
    if f.endswith((".jpg", ".jpeg", ".png"))
])

def visualize_prediction(image_path, model, class_names, conf_thresh=0.4):
    image = cv2.imread(image_path)
    result = model(image_path)[0]
    for box, score, label in zip(result.boxes.xyxy, result.boxes.conf, result.boxes.cls):
        if score < conf_thresh:
            continue
        x1, y1, x2, y2 = map(int, box.tolist())
        class_id = int(label)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(image, f"{class_names[class_id]} {score:.2f}",
                    (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

# 🔍 Exibe as primeiras 10 imagens com predições
for img_path in image_paths[:10]:
    print(f"🖼️ Imagem: {os.path.basename(img_path)}")
    visualize_prediction(img_path, model, dataset_yaml["names"], conf_thresh=0.4)

# 📈 Avaliação binária: classes presentes vs. ausentes
def evaluate_accuracy(model, image_paths, class_names):
    y_true_all, y_pred_all = [], []

    for img_path in tqdm(image_paths[:100]):
        label_path = img_path.replace("images", "labels").rsplit(".", 1)[0] + ".txt"
        if not os.path.exists(label_path):
            continue

        with open(label_path, "r") as f:
            gt_classes = [int(line.split()[0]) for line in f.readlines()]

        result = model(img_path)[0]
        pred_classes = result.boxes.cls.int().cpu().tolist() if result.boxes.cls is not None else []

        for cls in set(gt_classes + pred_classes):
            y_true_all.append(1 if cls in gt_classes else 0)
            y_pred_all.append(1 if cls in pred_classes else 0)

    precision = precision_score(y_true_all, y_pred_all, average="binary")
    recall = recall_score(y_true_all, y_pred_all, average="binary")
    f1 = f1_score(y_true_all, y_pred_all, average="binary")

    print("\n📊 Avaliação Binária:")
    print(f"✅ Precisão: {precision:.2f}")
    print(f"✅ Revocação: {recall:.2f}")
    print(f"✅ F1-score: {f1:.2f}")

# Chamada da avaliação
evaluate_accuracy(model, image_paths, dataset_yaml["names"])


Ultralytics 8.3.103 🚀 Python-3.12.9 torch-2.6.0 CPU (Apple M4 Pro)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=bdd100k_dataset.yaml, epochs=100, time=None, patience=10, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=cpu, workers=2, project=None, name=bdd100k_finetuned10, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True, l

train: Scanning /Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/Test/yolo_dataset/labels/train.cache... 1154 images, 1 backgrounds, 0 corrupt: 100%|██████████| 1154/1154 [00:00<?, ?it/s]
val: Scanning /Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/Test/yolo_dataset/labels/validation.cache... 10000 images, 1 backgrounds, 0 corrupt: 100%|██████████| 10000/10000 [00:00<?, ?it/s]


Plotting labels to runs/detect/bdd100k_finetuned10/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000714, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to runs/detect/bdd100k_finetuned10
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      1/100         0G      1.778      2.709      1.152         90        640: 100%|██████████| 73/73 [04:32<00:00,  3.74s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 313/313 [11:52<00:00,  2.28s/it]


                   all      10000     170805      0.617     0.0562      0.113     0.0646

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      2/100         0G      1.641       1.56      1.111         58        640: 100%|██████████| 73/73 [04:59<00:00,  4.10s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 313/313 [12:07<00:00,  2.32s/it]


                   all      10000     170805      0.547      0.158       0.14     0.0736

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      3/100         0G      1.618       1.45        1.1         99        640: 100%|██████████| 73/73 [04:31<00:00,  3.72s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95):  45%|████▌     | 141/313 [05:13<06:29,  2.27s/it]

In [ ]:
import os
import yaml
import cv2
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
from ultralytics import YOLO

# 📂 Configuração do dataset BDD100K
dataset_yaml = {
    "train": "/Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/Test/yolo_dataset/images/train",
    "val": "/Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/Test/yolo_dataset/images/validation",
    "test": "/Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/Test/yolo_dataset/images/test",
    "nc": 10,
    "names": [
        "traffic sign", "traffic light", "car", "rider", "motor",
        "person", "bus", "truck", "bike", "train"
    ]
}

# 📝 Salva em dataset.yaml
with open("bdd100k_dataset.yaml", "w") as f:
    yaml.dump(dataset_yaml, f)

# 🔍 Detecta se há GPU disponível
device = "0" if torch.cuda.is_available() else "cpu"
print(f"💻 Usando dispositivo: {'GPU' if device == '0' else 'CPU'}")

# 🚀 Treinamento YOLOv8
model = YOLO("yolov8n.pt")  # ou yolov8m.pt, yolov8s.pt...

model.train(
    data="bdd100k_dataset.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    name="bdd100k_finetuned",
    workers=2,
    patience=10,
    device=device,  # 👈 GPU (0) ou CPU automático
    verbose=True
)

# 📍 Carregar melhor modelo treinado
best_model_path = "runs/detect/bdd100k_finetuned/weights/best.pt"
model = YOLO(best_model_path)

# 📊 Avaliação no conjunto de validação
results = model.val(data="bdd100k_dataset.yaml", split="val", save_json=True)

# 🧪 Visualização de predições
val_dir = dataset_yaml["val"]
image_paths = sorted([
    os.path.join(val_dir, f) for f in os.listdir(val_dir)
    if f.endswith((".jpg", ".jpeg", ".png"))
])

def visualize_prediction(image_path, model, class_names, conf_thresh=0.4):
    image = cv2.imread(image_path)
    result = model(image_path)[0]
    for box, score, label in zip(result.boxes.xyxy, result.boxes.conf, result.boxes.cls):
        if score < conf_thresh:
            continue
        x1, y1, x2, y2 = map(int, box.tolist())
        class_id = int(label)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(image, f"{class_names[class_id]} {score:.2f}",
                    (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

# 🔍 Exibe as primeiras 10 imagens com predições
for img_path in image_paths[:10]:
    print(f"🖼️ Imagem: {os.path.basename(img_path)}")
    visualize_prediction(img_path, model, dataset_yaml["names"], conf_thresh=0.4)

# 📈 Avaliação binária: classes presentes vs. ausentes
def evaluate_accuracy(model, image_paths, class_names):
    y_true_all, y_pred_all = [], []

    for img_path in tqdm(image_paths[:100]):
        label_path = img_path.replace("images", "labels").rsplit(".", 1)[0] + ".txt"
        if not os.path.exists(label_path):
            continue

        with open(label_path, "r") as f:
            gt_classes = [int(line.split()[0]) for line in f.readlines()]

        result = model(img_path)[0]
        pred_classes = result.boxes.cls.int().cpu().tolist() if result.boxes.cls is not None else []

        for cls in set(gt_classes + pred_classes):
            y_true_all.append(1 if cls in gt_classes else 0)
            y_pred_all.append(1 if cls in pred_classes else 0)

    precision = precision_score(y_true_all, y_pred_all, average="binary")
    recall = recall_score(y_true_all, y_pred_all, average="binary")
    f1 = f1_score(y_true_all, y_pred_all, average="binary")

    print("\n📊 Avaliação Binária (por classe presente/ausente):")
    print(f"✅ Precisão: {precision:.2f}")
    print(f"✅ Revocação: {recall:.2f}")
    print(f"✅ F1-score: {f1:.2f}")

# Chamada da avaliação
evaluate_accuracy(model, image_paths, dataset_yaml["names"])
